In [11]:
import pathlib
import random
import copy
import numpy as np
import torch
#import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft


from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [13]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41,42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [15]:
def load_data(subject_index):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
        
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    device = torch.device("cpu")
    input_shape_st = (60, 900)
        
    return all_epochs, labels_raw, ch_names, device, input_shape_st


In [32]:
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

def compute_connectivity(eeg_data, ch_names, sfreq=1000, window_size=256, overlap=0.5, return_dataframe=False):
    """
    Compute coherence and lagged coherence for EEG data.
    
    Parameters:
    -----------
    eeg_data : numpy.ndarray
        EEG data with shape [trials, channels, time]
    ch_names : list
        List of channel names corresponding to channel indices
    sfreq : float
        Sampling frequency in Hz
    window_size : int
        Size of the window for spectral estimation
    overlap : float
        Overlap between windows (0 to 1)
    return_dataframe : bool, default=False
        If True, return non-averaged results as a pandas DataFrame
        
    Returns:
    --------
    If return_dataframe=False:
        coherence : dict
            Dictionary of coherence matrices for each frequency band
        lagged_coherence : dict
            Dictionary of lagged coherence matrices for each frequency band
    If return_dataframe=True:
        df : pandas.DataFrame
            DataFrame containing non-averaged results with columns:
            trial, ch_name1, ch_name2, ch_index1, ch_index2, and coherence/lagged coherence values
            for each frequency band
    """
    # Extract dimensions
    n_trials, n_channels, n_times = eeg_data.shape
    
    # Define frequency bands
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }
    
    # Initialize result matrices
    coherence = {band: np.zeros((n_channels, n_channels)) for band in bands}
    lagged_coherence = {band: np.zeros((n_channels, n_channels)) for band in bands}
    
    # Calculate parameters for Welch's method
    nperseg = window_size
    noverlap = int(window_size * overlap)
    
    # If returning a DataFrame, prepare list to store results
    if return_dataframe:
        results = []
    
    # Process each trial
    for trial in range(n_trials):
        # Arrays to store auto- and cross-spectra
        freqs = None  # Will store frequency bins
        auto_spectra = np.zeros((n_channels, nperseg//2+1))
        cross_spectra_real = np.zeros((n_channels, n_channels, nperseg//2+1))
        cross_spectra_imag = np.zeros((n_channels, n_channels, nperseg//2+1))
        
        # First pass: compute auto-spectra for all channels
        for ch in range(n_channels):
            # Use Welch's method for better spectral estimation
            f, Pxx = signal.welch(
                eeg_data[trial, ch], 
                fs=sfreq, 
                nperseg=nperseg, 
                noverlap=noverlap,
                return_onesided=True
            )
            
            if freqs is None:
                freqs = f  # Store frequency bins
                
            auto_spectra[ch] = Pxx
            
        # Second pass: compute cross-spectra for all channel pairs
        for i in range(n_channels):
            for j in range(i, n_channels):
                if i == j:
                    # No need to compute cross-spectrum for same channel
                    continue
                    
                # Compute cross-spectral density
                f, Pxy = signal.csd(
                    eeg_data[trial, i], 
                    eeg_data[trial, j], 
                    fs=sfreq, 
                    nperseg=nperseg, 
                    noverlap=noverlap,
                    return_onesided=True
                )
                
                # Store real and imaginary parts separately
                cross_spectra_real[i, j] = np.real(Pxy)
                cross_spectra_imag[i, j] = np.imag(Pxy)
                
                # Mirror for computational efficiency
                cross_spectra_real[j, i] = cross_spectra_real[i, j]
                cross_spectra_imag[j, i] = -cross_spectra_imag[i, j]  # Conjugate
        
        # Calculate coherence measures for each frequency band
        for band, (band_min, band_max) in bands.items():
            # Find frequency indices within the band
            freq_idx = np.where((freqs >= band_min) & (freqs <= band_max))[0]
            
            if len(freq_idx) == 0:
                continue  # Skip if no frequencies in this band
                
            # Calculate measures for each channel pair
            for i in range(n_channels):
                for j in range(i+1, n_channels):  # Skip diagonal (i==j)
                    # Get auto-spectra for both channels
                    Pxx = auto_spectra[i, freq_idx]
                    Pyy = auto_spectra[j, freq_idx]
                    
                    # Get cross-spectrum components
                    Pxy_real = cross_spectra_real[i, j, freq_idx]
                    Pxy_imag = cross_spectra_imag[i, j, freq_idx]
                    
                    # Calculate |Pxy|²
                    Pxy_abs_squared = Pxy_real**2 + Pxy_imag**2
                    
                    # Calculate denominator for both measures
                    denominator = Pxx * Pyy
                    
                    # Avoid division by zero
                    valid_idx = denominator > 0
                    
                    if np.any(valid_idx):
                        # Standard coherence: |Pxy|² / (Pxx * Pyy)
                        coh = np.zeros_like(Pxy_abs_squared)
                        coh[valid_idx] = Pxy_abs_squared[valid_idx] / denominator[valid_idx]
                        
                        # Lagged coherence: Im(Pxy)² / (Pxx * Pyy - Re(Pxy)²)
                        lagged_num = Pxy_imag**2
                        lagged_denom = denominator - Pxy_real**2
                        
                        lagged_valid_idx = lagged_denom > 0
                        lagged_coh = np.zeros_like(lagged_num)
                        
                        if np.any(lagged_valid_idx):
                            lagged_coh[lagged_valid_idx] = lagged_num[lagged_valid_idx] / lagged_denom[lagged_valid_idx]
                        
                        # Average across frequency band
                        avg_coh = np.mean(coh)
                        avg_lagged_coh = np.mean(lagged_coh)
                        
                        if return_dataframe:
                            # Store in results list for DataFrame
                            results.append({
                                'trial': trial,
                                'ch_name1': ch_names[i],
                                'ch_name2': ch_names[j],
                                'ch_index1': i,
                                'ch_index2': j,
                                'frequency_band': band,
                                'coherence': avg_coh,
                                'lagged_coherence': avg_lagged_coh
                            })
                        else:
                            # Store in result matrices (divide by n_trials for averaging)
                            coherence[band][i, j] += avg_coh / n_trials
                            coherence[band][j, i] += avg_coh / n_trials  # Mirror
                            
                            lagged_coherence[band][i, j] += avg_lagged_coh / n_trials
                            lagged_coherence[band][j, i] += avg_lagged_coh / n_trials  # Mirror
    
    if return_dataframe:
        # Create DataFrame from results list
        df = pd.DataFrame(results)
        return df
    else:
        return coherence, lagged_coherence

def plot_connectivity_matrix(connectivity, band, ch_names=None, title=None, cmap='viridis', vmin=0, vmax=1):
    """
    Plot connectivity matrix as a heatmap.
    
    Parameters:
    -----------
    connectivity : dict
        Dictionary of connectivity matrices for each frequency band
    band : str
        Frequency band to plot
    ch_names : list, optional
        List of channel names for axis labels
    title : str, optional
        Title for the plot
    cmap : str, default='viridis'
        Colormap for the heatmap
    vmin, vmax : float, default=0, 1
        Minimum and maximum values for color scaling
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The generated figure
    """
    plt.figure(figsize=(10, 8))
    im = plt.imshow(connectivity[band], cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, label='Connectivity strength')
    
    if title:
        plt.title(f"{title} - {band.capitalize()} band")
    else:
        plt.title(f"Connectivity - {band.capitalize()} band")
    
    # Add channel names as tick labels if provided
    if ch_names is not None:
        n_channels = len(ch_names)
        
        # Add tick positions and labels for a subset of channels if there are many
        if n_channels > 20:
            tick_step = max(1, n_channels // 10)
            tick_positions = np.arange(0, n_channels, tick_step)
            tick_labels = [ch_names[i] for i in tick_positions]
        else:
            tick_positions = np.arange(n_channels)
            tick_labels = ch_names
            
        plt.xticks(tick_positions, tick_labels, rotation=90, fontsize=8)
        plt.yticks(tick_positions, tick_labels, fontsize=8)
    else:
        plt.xlabel('Channel')
        plt.ylabel('Channel')
    
    plt.tight_layout()

def plot_connectivity_from_dataframe(df, measure_type='coherence', band=None, trial=None, 
                                    ch_names=None, title=None, cmap='viridis', vmin=0, vmax=1):
    """
    Plot connectivity matrix from DataFrame results.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with connectivity results
    measure_type : str, default='coherence'
        Type of measure to plot ('coherence' or 'lagged_coherence')
    band : str, optional
        Frequency band to plot (if None, will use the first band found)
    trial : int, optional
        Trial to plot (if None, will average across all trials)
    ch_names : list, optional
        List of channel names for axis labels
    title : str, optional
        Title for the plot
    cmap : str, default='viridis'
        Colormap for the heatmap
    vmin, vmax : float, default=0, 1
        Minimum and maximum values for color scaling
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The generated figure
    """
    # Filter DataFrame by band if specified
    if band is not None:
        df_filtered = df[df['frequency_band'] == band]
        if df_filtered.empty:
            raise ValueError(f"No data found for frequency band '{band}'")
        band_name = band
    else:
        # Use the first band in the DataFrame
        band_name = df['frequency_band'].iloc[0]
        df_filtered = df[df['frequency_band'] == band_name]
    
    # Filter by trial if specified
    if trial is not None:
        df_filtered = df_filtered[df_filtered['trial'] == trial]
        if df_filtered.empty:
            raise ValueError(f"No data found for trial {trial}")
    
    # Get unique channel indices and names
    ch_indices = sorted(list(set(df_filtered['ch_index1'].tolist() + df_filtered['ch_index2'].tolist())))
    if ch_names is None:
        ch_names = sorted(list(set(df_filtered['ch_name1'].tolist() + df_filtered['ch_name2'].tolist())))
    
    n_channels = len(ch_indices)
    
    # Create empty connectivity matrix
    connectivity = np.zeros((n_channels, n_channels))
    
    # Group by channel pairs and average if needed
    if trial is None:
        avg_df = df_filtered.groupby(['ch_index1', 'ch_index2'])[measure_type].mean().reset_index()
    else:
        avg_df = df_filtered[['ch_index1', 'ch_index2', measure_type]]
    
    # Fill connectivity matrix with values
    for _, row in avg_df.iterrows():
        i, j = int(row['ch_index1']), int(row['ch_index2'])
        connectivity[i, j] = row[measure_type]
        connectivity[j, i] = row[measure_type]  # Mirror for symmetric matrix
    
    # Create figure
    plt.figure(figsize=(10, 8))
    im = plt.imshow(connectivity, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, label=f'{measure_type.replace("_", " ").title()} strength')
    
    # Set title
    if title:
        if trial is not None:
            plt.title(f"{title} - {band_name.capitalize()} band - Trial {trial}")
        else:
            plt.title(f"{title} - {band_name.capitalize()} band - Average")
    else:
        if trial is not None:
            plt.title(f"{measure_type.replace('_', ' ').title()} - {band_name.capitalize()} band - Trial {trial}")
        else:
            plt.title(f"{measure_type.replace('_', ' ').title()} - {band_name.capitalize()} band - Average")
    
    # Add channel names as tick labels if provided
    if ch_names is not None:
        # Add tick positions and labels for a subset of channels if there are many
        if n_channels > 20:
            tick_step = max(1, n_channels // 10)
            tick_positions = np.arange(0, n_channels, tick_step)
            tick_labels = [ch_names[i] for i in tick_positions]
        else:
            tick_positions = np.arange(n_channels)
            tick_labels = ch_names
            
        plt.xticks(tick_positions, tick_labels, rotation=90, fontsize=8)
        plt.yticks(tick_positions, tick_labels, fontsize=8)
    
    plt.tight_layout()


# Example usage:
# eeg_data shape: [trials, channels, time_points]
# ch_names = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', ...]  # Channel names
#
# # Get averaged results as matrices
# coherence, lagged_coherence = compute_connectivity(eeg_data, ch_names, sfreq=1000)
# 
# # Plot alpha band coherence
# plot_connectivity_matrix(coherence, 'alpha', ch_names=ch_names, title='Coherence')
# plt.show()
#
# # Plot alpha band lagged coherence
# plot_connectivity_matrix(lagged_coherence, 'alpha', ch_names=ch_names, title='Lagged Coherence')
# plt.show()
#
# # Alternatively, get non-averaged results as a DataFrame
# results_df = compute_connectivity(eeg_data, ch_names, sfreq=1000, return_dataframe=True)
#
# # Example: Show the first few rows
# print(results_df.head())
#
# # Example: Filter DataFrame for specific bands, channels or trials
# alpha_results = results_df[results_df['frequency_band'] == 'alpha']
# frontal_results = results_df[(results_df['ch_name1'].str.contains('F')) & 
#                             (results_df['ch_name2'].str.contains('F'))]
# 
# # Example: Get average coherence per channel pair across trials
# avg_per_pair = results_df.groupby(['ch_name1', 'ch_name2', 'frequency_band'])['coherence'].mean().reset_index()
#
# # Example: Plot connectivity matrix for a specific trial and band
# plot_connectivity_from_dataframe(results_df, measure_type='coherence', band='alpha', trial=0, 
#                                 ch_names=ch_names, title='Alpha Coherence')
# plt.show()
#
# # Example: Create a pivot table for easier data manipulation
# pivot_table = results_df.pivot_table(
#     index=['trial', 'ch_name1', 'ch_name2'], 
#     columns='frequency_band', 
#     values=['coherence', 'lagged_coherence']
# ).reset_index()
#
# # Example: Save results to CSV
# results_df.to_csv('eeg_connectivity_results.csv', index=False)

In [33]:
all_epochs, labels_raw, ch_names, device, input_shape_st = load_data(2)

wandb:
  key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model: null
dataset:
  data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
  file_name: subject_{:03d}_preprocessed_combined_py.fif
  exclude_timepoints: 100
  subject_index: 2
  test_subject_indices:
  - 1
  - 2
  - 13
  - 24
  - 26
  - 27
  - 29
  - 34
  - 35
  - 41
  - 42
  - 43
  - 45
  - 46
  - 47
  - 48
  - 52
  - 55
  - 56
  - 57
  - 60
  - 62
  - 67
  - 69
  - 72
  - 73
  - 79
  - 80
  - 86
  - 88
  - 92
  - 102
training:
  training_start_len: 100
  pretrain_epochs: 100
  pretrain_lr: 0.0001
  val_window_len: 1
  epochs_per_window: 10
  num_warmup_epochs: 0
  num_epochs: 800
  slide_step: 1
  num_warmup_epochs_per_window: 0
  lr: 0.005
  nll_beta: 0.001
  batch_size: 50
  random_seed: 42
  precision: bf16
  kde_lambda: 0.5
  finetune_entire_model: true
exp_name: S4_S4EEGNet_ema_100_cal_py_2_

Loading EEG data...

subject index: 2, frequency band: None

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/dat

/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [34]:
df = compute_connectivity(all_epochs, ch_names,sfreq=1000, return_dataframe=True)

In [37]:
df.to_csv('connectivity_results.csv', index=False)

In [38]:
cfg = load_config()

In [40]:
dir = "coherence_results"
os.makedirs(dir, exist_ok=True)
for subject_index in cfg.dataset.test_subject_indices:
    all_epochs, labels_raw, ch_names, device, input_shape_st = load_data(subject_index)
    df = compute_connectivity(all_epochs, ch_names,sfreq=1000, return_dataframe=True)
    df.to_csv(f'{dir}/coherence_results_{subject_index}.csv', index=False)

wandb:
  key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model: null
dataset:
  data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
  file_name: subject_{:03d}_preprocessed_combined_py.fif
  exclude_timepoints: 100
  subject_index: 1
  test_subject_indices:
  - 1
  - 2
  - 13
  - 24
  - 26
  - 27
  - 29
  - 34
  - 35
  - 41
  - 42
  - 43
  - 45
  - 46
  - 47
  - 48
  - 52
  - 55
  - 56
  - 57
  - 60
  - 62
  - 67
  - 69
  - 72
  - 73
  - 79
  - 80
  - 86
  - 88
  - 92
  - 102
training:
  training_start_len: 100
  pretrain_epochs: 100
  pretrain_lr: 0.0001
  val_window_len: 1
  epochs_per_window: 10
  num_warmup_epochs: 0
  num_epochs: 800
  slide_step: 1
  num_warmup_epochs_per_window: 0
  lr: 0.005
  nll_beta: 0.001
  batch_size: 50
  random_seed: 42
  precision: bf16
  kde_lambda: 0.5
  finetune_entire_model: true
exp_name: S4_S4EEGNet_ema_100_cal_py_1_

Loading EEG data...

subject index: 1, frequency band: None

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/dat

    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
wandb:
  key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model: null
dataset:
  data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
  file_name: subject_{:03d}_preprocessed_combined_py.fif
  exclude_timepoints: 100
  subject_index: 2
  test_subject_indices:
  - 1
  - 2
  - 13
  - 24
  - 26
  - 27
  - 29
  - 34
  - 35
  - 41
  - 42
  - 43
  - 45
  - 46
  - 47
  - 48
  - 52
  - 55
  - 56
  - 57
  - 60
  - 62
  - 67
  - 69
  - 72
  - 73
  - 79
  - 80
  - 86
  - 88
  - 92
  - 102
training:
  training_start_len: 100
  pretrain_epochs: 100
  pretrain_lr: 0.0001
  val_window_len: 1
  epochs_per_window: 10
  num_warmup_epochs: 0
  num_epochs: 800
  slide_step: 1
  num_warmup_epochs_per_window: 0
  lr: 0.005
  nll_beta: 0.001
  batch_size: 50
  random_seed: 42
  precision: bf16
  kde_lambda: 0.5
  finetune_entire_model: true
exp_name: S4_S4E

/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


...

Adding metadata with 1 columns
608 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
wandb:
  key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model: null
dataset:
  data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
  file_name: subject_{:03d}_preprocessed_combined_py.fif
  exclude_timepoints: 100
  subject_index: 92
  test_subject_indices:
  - 1
  - 2
  - 13
  - 24
  - 26
  - 27
  - 29
  - 34
  - 35
  - 41
  - 42
  - 43
  - 45
  - 46
  - 47
  - 48
  - 52
  - 55
  - 56
  - 57
  - 60
  - 62
  - 67
  - 69
  - 72
  - 73
  - 79
  - 80
  - 86
  - 88
  - 92
  - 102
training:
  training_start_len: 100
  pretrain_epochs: 100
  pretrain_lr: 0.0001
  val_window_len: 1
  epochs_per_window: 10
  num_warmup_epochs: 0
  num_epochs: 800
  slide_step: 1
  num_warmup_epochs_per_window: 0
  lr: 0.005
  nll_beta: 0.001
  batch_size: 50
  random_seed: 42
  precision: bf16
  kde_lambda: 0.5
  finetune_entire_model: true
exp_name: S4_S4

/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_092_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
564 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
wandb:
  key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model: null
dataset:
  data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
  file_name: subject_{:03d}_preprocessed_combined_py.fif
  exclude_timepoints: 100
  subject_index: 102
  test_subject_indices:
  - 1
  - 2
  - 13
  - 24
  - 26
  - 27
  - 29
  - 34
  - 35
  - 41
  - 42
  - 43
  - 45
  - 46
  - 47
  - 48
  - 52
  - 55
  - 56
  - 57
  - 60
  - 62
  - 67
  - 69
  - 72
  - 73
  - 79
  - 80
  - 86
  - 88
  - 92
  - 102
training:
  training_start_len: 100
  pretrain_epochs: 100
  pretrain_lr: 0.0001
  val_window_len: 1
  epochs_per_window: 10
  num_warmup_epochs: 0
  num_epochs: 800
  slide_step: 1
  num_warmup_epochs_per_window: 0
  lr: 0.005
  nll_beta: 0.001
  batch_size: 50
  random_seed: 42
  precision: bf16
  kde_lambda: 0.5
  finetune_entire_model: true
exp_name: S4_S

/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


coherence